# PaHaW Mamba Baseline

This notebook loads the raw-sequence Mamba baseline config and trains it on the PaHaW dataset.

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "config").exists() and (candidate / "src").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate the repo root from the current working directory.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.common.utils import build_pahaw_mapping, load_ruamel, set_determinism
from src.data.pahaw_dataset import PaHaWDataset
from src.model.mamba_sequence_classifier import MambaSequenceClassifier
from src.run.model.run_sequence_classifier import RunSequenceClassifier
from src.visualization.vis_enum import PaHaWFeatures, PaHaWTasks

print(f"Repo root: {REPO_ROOT}")

Repo root: /home/walter/projects/parkinson_disease


In [2]:
general_cfg = load_ruamel(REPO_ROOT / "config/general.yaml")
preproc_cfg = load_ruamel(REPO_ROOT / "config/preprocessing/general.yaml")
mamba_cfg = load_ruamel(REPO_ROOT / "config/model/mamba_experiment_sequence.yaml")

model_cfg = dict(mamba_cfg["model"])
training_cfg = dict(mamba_cfg["training"])
base_experiment_cfg = dict(mamba_cfg["experiments"][0])

def task_slug(task: PaHaWTasks) -> str:
    return task.name.lower()

experiment_cfgs = []
selected_tasks = [
    PaHaWTasks.SENTENCE_TRAMVAJ_DNES_UZ_NEPOJEDE,
    PaHaWTasks.WORD_NEPOPADNOUT,
    PaHaWTasks.WORD_POROVNAT,
]

for task in selected_tasks:
    experiment_cfg = dict(base_experiment_cfg)
    train_slug = task_slug(task)
    val_slug = task_slug(task)
    experiment_cfg["name"] = f"train_on_{train_slug}_validate_on_{val_slug}"
    experiment_cfg["general_type_of_input"] = "raw_sequence"
    experiment_cfg["specific_type_of_input"] = f"all_features/train_on_{train_slug}_validate_on_{val_slug}"
    experiment_cfg["train_task_numbers"] = [task.value + 1]
    experiment_cfg["val_task_numbers"] = [task.value + 1]
    experiment_cfgs.append(experiment_cfg)

# Optional quick overrides before training.
training_cfg["epochs"] = 60
training_cfg["batch_size"] = 8
training_cfg["device"] = "cuda:0"
training_cfg["log_every_n_batches"] = 8

print("Model config:", model_cfg)
print("Training config:", training_cfg)
print("Experiments:", [experiment["name"] for experiment in experiment_cfgs])

Model config: {'num_classes': 2, 'd_model': 64, 'n_layers': 2, 'd_state': 16, 'expand': 2, 'd_conv': 4, 'dropout': 0.2}
Training config: {'experiments_root': 'experiments', 'k_folds': 5, 'epochs': 60, 'batch_size': 8, 'learning_rate': 0.0005, 'num_workers': 0, 'device': 'cuda:0', 'normalize_sequences': True, 'stats_batch_size': 8, 'log_every_n_batches': 8, 'save_metrics': True, 'results_writer': 'csv', 'metrics': ['accuracy', 'f1']}
Experiments: ['train_on_sentence_tramvaj_dnes_uz_nepojede_validate_on_sentence_tramvaj_dnes_uz_nepojede', 'train_on_word_nepopadnout_validate_on_word_nepopadnout', 'train_on_word_porovnat_validate_on_word_porovnat']


In [3]:
set_determinism(general_cfg["seed"])

feature_names = [feature.value for feature in PaHaWFeatures]
mapping = build_pahaw_mapping(str(REPO_ROOT / general_cfg["base_dir"]), feature_names)
df = pd.read_excel(REPO_ROOT / general_cfg["csv_file"], engine="openpyxl")

dataset = PaHaWDataset(mapping=mapping, df=df)
excluded = set(preproc_cfg["global"].get("ids_to_exclude", []))
valid_patients = [patient_id for patient_id in sorted(mapping.keys()) if patient_id not in excluded]

print(f"Total patients in mapping: {len(mapping)}")
print(f"Excluded patients: {sorted(excluded)}")
print(f"Valid patients: {len(valid_patients)}")

Total patients in mapping: 75
Excluded patients: ['00061', '00080', '00089']
Valid patients: 72


In [ ]:
pipeline = RunSequenceClassifier(
    model_class=MambaSequenceClassifier,
    dataset=dataset,
    valid_patients=valid_patients,
    experiment_cfgs=experiment_cfgs,
    model_cfg=model_cfg,
    training_cfg=training_cfg,
    seed=general_cfg["seed"],
)

pipeline.run()


[RunSequenceClassifier] Starting experiment: train_on_sentence_tramvaj_dnes_uz_nepojede_validate_on_sentence_tramvaj_dnes_uz_nepojede
  [RunSequenceClassifier] Fold 1/5
    [RunSequenceClassifier] Normalization stats mean=[4492.6341388659675, 1713.2510432629965, 96586199.07664742, 0.6637746080396086, 3091.2017210892373, 562.7409642815041, 573.0566544854415] std=[1842.0526450295602, 602.3017143489291, 11220633909.795067, 0.47241705913469356, 304.35003440376596, 72.93370359600661, 507.63301407036795]
    [RunSequenceClassifier] Train batch 8/8 | loss: 1.5572 | accuracy: 0.0000 | f1: 0.0000 | elapsed: 10.6s
    [RunSequenceClassifier] Val batch 2/2 | loss: 0.5729 | accuracy: 0.5714 | f1: 0.0000 | elapsed: 0.5s
  Epoch 01/60 | Train loss: 0.9276 | Val loss: 0.6515 | Train accuracy: 0.4912 | Train f1: 0.0000 | Val accuracy: 0.5333 | Val f1: 0.0000
    [RunSequenceClassifier] Best accuracy checkpoint saved -> /home/walter/projects/parkinson_disease/experiments/mamba_sequence_classifier/raw_

In [ ]:
results_root = (
    REPO_ROOT
    / training_cfg.get("experiments_root", "experiments")
    / "mamba_sequence_classifier"
    / "raw_sequence"
)

summary_frames = []
for experiment in experiment_cfgs:
    results_dir = results_root / experiment["specific_type_of_input"]
    summary_df = pd.read_csv(results_dir / "summary.csv")
    summary_df.insert(0, "experiment_name", experiment["name"])
    summary_frames.append(summary_df)

all_results_df = pd.concat(summary_frames, ignore_index=True)
display(all_results_df)

In [ ]:
###Reimplement the baseline from papers